<a href="https://colab.research.google.com/github/Lucas66666677/xhs-jpg-models/blob/main/%E5%85%A8%E8%87%AA%E5%8B%95%E5%B0%8F%E7%B4%85%E6%9B%B8%E7%94%9F%E5%9C%96%E7%A5%9E%E5%99%A8%EF%BC%8C%E6%A8%A1%E6%9D%BF1%EF%BC%88V9_8).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 環境設定、字庫、文字換行與分頁算法

In [ ]:
# ==========================================
# 🛑 模块零：安全配置与核心套件
# ==========================================
import os
import re
import time
import requests
import json
import ipywidgets as widgets
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display, Image as IPImage, clear_output

# 🌟 安全升级：从 Colab Secrets 环境中读取 Key，GitHub 上将看不到真实数值
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    HF_TOKEN = userdata.get('HF_TOKEN')
    print("✅ 已安全载入 API 密钥。")
except Exception as e:
    print("⚠️ 未能在 Secrets 中找到密钥，请确保已在 Colab 左侧『钥匙』图标处设定 GOOGLE_API_KEY 与 HF_TOKEN。")
    GOOGLE_API_KEY = "SECRET_NOT_FOUND"
    HF_TOKEN = "SECRET_NOT_FOUND"

# 字体路径设定
FONT_PATH = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
BOLD_PATH = "/usr/share/fonts/opentype/noto/NotoSansCJK-Bold.ttc"

# ==========================================
# 🛠️ 模块一：环境与字库初始化
# ==========================================
def setup_environment():
    """安装中文字体并初始化字体对象 (简体纯净模式 index=2)"""
    if not os.path.exists(FONT_PATH):
        print("📥 正在安装系统级中文字体包...")
        os.system("apt-get -qq update && apt-get -qq install -y fonts-noto-cjk")

    fonts = {
        'title': ImageFont.truetype(BOLD_PATH, 115, index=2),
        'title_s': ImageFont.truetype(BOLD_PATH, 48, index=2),
        'sub': ImageFont.truetype(BOLD_PATH, 42, index=2),
        'stats': ImageFont.truetype(FONT_PATH, 40, index=2),
        'body': ImageFont.truetype(FONT_PATH, 45, index=2),
        'footer': ImageFont.truetype(FONT_PATH, 32, index=2)
    }
    return fonts

# ==========================================
# 🧠 模块二：像素级视觉换行
# ==========================================
def visual_wrap(text, font, max_px_width):
    """计算中英文字符的真实像素宽度，避免标点符号落在行首"""
    lines = []
    current_line = ""
    punc_list = "。，！？）》】」』；：、"
    for char in text:
        test_line = current_line + char
        bbox = font.getbbox(test_line)
        line_w = bbox[2] - bbox[0]
        if line_w > max_px_width:
            if char in punc_list and len(current_line) > 0:
                lines.append(current_line[:-1])
                current_line = current_line[-1] + char
            else:
                lines.append(current_line)
                current_line = char
        else:
            current_line = test_line
    if current_line: lines.append(current_line)
    return lines

# ==========================================
# 🧠 模块三：核心分页算法 (全域原子化不拆段)
# ==========================================
def paginate_content(processed_content, fonts):
    """计算每一行文字的 Y 坐标，控制段落间距，并自动换页"""
    start_y = 190
    h_limit = 1440 - 120
    pages_data = []
    current_page = []
    y = start_y
    line_h_para, line_h_list = 80, 75
    paragraph_gap = 45
    inner_w = 1080 - 160

    for item in processed_content:
        # 预计算排版宽度与前缀
        if item['type'] == 'num_list':
            prefix = f"{int(item['num']):02d}."
            p_w = 110
            lines = visual_wrap(item['text'], fonts['body'], inner_w - p_w)
            line_h = line_h_list
        elif item['type'] == 'bullet_list':
            p_w = 45
            lines = visual_wrap(item['text'], fonts['body'], inner_w - p_w)
            line_h = line_h_list
        else:
            p_w = 0
            lines = visual_wrap(item['text'], fonts['body'], inner_w)
            line_h = line_h_para

        total_block_h = len(lines) * line_h

        # 核心逻辑：如果这一页塞不下这一整段，就整体移到下一页
        if y + total_block_h > h_limit and total_block_h < (h_limit - start_y):
            if current_page:
                pages_data.append(current_page)
                current_page = []
                y = start_y

        for idx, line in enumerate(lines):
            if y + line_h > h_limit:
                if current_page: pages_data.append(current_page)
                current_page = []
                y = start_y
            current_page.append({
                'type': item['type'], 'text': line, 'y': y,
                'is_first_line': (idx == 0),
                'prefix': prefix if (idx == 0 and item['type'] == 'num_list') else '',
                'indent': p_w
            })
            y += line_h

        y += paragraph_gap # 段落间的物理呼吸空间

    if current_page: pages_data.append(current_page)
    return pages_data

✅ 已安全载入 API 密钥。


## 視覺渲染

In [ ]:
# ==========================================
# 📐 模块四：像素级排版引擎 (🌟 终极去毒 + 呼吸感优化版)
# ==========================================
# 🌟 核心修复：1. 清洗正则匹配组，彻底剥除 Markdown！ 2. 增加段落间距！ 3. 解决副标题与底图重叠问题！
import os
import re
from PIL import Image, ImageDraw, ImageFont

FONT_PATH = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
BOLD_PATH = "/usr/share/fonts/opentype/noto/NotoSansCJK-Bold.ttc"

def setup_environment():
    if not os.path.exists(FONT_PATH):
        print("📥 正在安装系统级中文字体包...")
        os.system("apt-get -qq update && apt-get -qq install -y fonts-noto-cjk")
    return {'title': BOLD_PATH, 'content': BOLD_PATH, 'sub': FONT_PATH, 'num': FONT_PATH}

def visual_wrap(text, font, max_px_width):
    lines = []
    current_line = ""
    punc_list = "。，！？）》】」』；：、"
    for char in text:
        test_line = current_line + char
        bbox = font.getbbox(test_line)
        line_w = bbox[2] - bbox[0]
        if line_w > max_px_width:
            if char in punc_list and len(current_line) > 0:
                lines.append(current_line[:-1])
                current_line = current_line[-1] + char
            else:
                lines.append(current_line)
                current_line = char
        else:
            current_line = test_line
    if current_line: lines.append(current_line)
    return lines

def paginate_content(processed_content, fonts):
    start_y = 190
    h_limit = 1440 - 120
    pages_data = []
    current_page = []
    y = start_y
    line_h_para, line_h_list = 80, 75
    # 🌟 呼吸感优化：增加段落间距（从 45 -> 60 px）
    paragraph_gap = 60
    inner_w = 1080 - 160

    for item in processed_content:
        if item['type'] == 'sub_heading':
            if current_page:
                pages_data.append(current_page)
                current_page = []
                y = start_y
            p_w = 0
            lines = visual_wrap(item['text'], fonts['title_s'], inner_w)
            line_h = 90
        elif item['type'] == 'num_list':
            prefix = f"{int(item['num']):02d}."
            p_w = 110
            lines = visual_wrap(item['text'], fonts['body'], inner_w - p_w)
            line_h = line_h_list
        elif item['type'] == 'bullet_list':
            p_w = 45
            lines = visual_wrap(item['text'], fonts['body'], inner_w - p_w)
            line_h = line_h_list
        else:
            p_w = 0
            lines = visual_wrap(item['text'], fonts['body'], inner_w)
            line_h = line_h_para

        total_block_h = len(lines) * line_h

        if y + total_block_h > h_limit and total_block_h < (h_limit - start_y):
            if current_page:
                pages_data.append(current_page)
                current_page = []
                y = start_y

        for idx, line in enumerate(lines):
            if y + line_h > h_limit:
                if current_page: pages_data.append(current_page)
                current_page = []
                y = start_y

            current_page.append({
                'type': item['type'], 'text': line, 'y': y,
                'is_first_line': (idx == 0),
                'prefix': prefix if (idx == 0 and item['type'] == 'num_list') else '',
                'indent': p_w
            })
            y += line_h
        y += paragraph_gap

    if current_page: pages_data.append(current_page)
    return pages_data

def draw_cover(data, photo_path, fonts):
    w, h = 1080, 1440
    canvas = Image.new('RGB', (w, h), color=(255, 255, 255))
    draw = ImageDraw.Draw(canvas)
    margin = margin_h = 80

    draw.text((margin, 60), "LUCAS 拆书实验室 | 拒绝无效内耗", font=fonts['footer'], fill=(120, 120, 120))
    draw.rectangle([margin, 130, margin + 60, 138], fill=(230, 50, 50))
    y = 200

    title_text = data.get('title') or '无标题'
    for line in visual_wrap(title_text, fonts['title'], w - 160):
        bbox = draw.textbbox((0, 0), line, font=fonts['title'])
        draw.text(((w - (bbox[2] - bbox[0])) // 2, y), line, font=fonts['title'], fill=(0, 0, 0))
        y += (bbox[3] - bbox[1]) + 25

    y = max(y + 30, 450)
    stats = f" WORDS: {data.get('words', '1000')} | TIME: {data.get('time', '4 mins')} "
    s_bbox = draw.textbbox((0, 0), stats, font=fonts['stats'])
    bw, bh = s_bbox[2] - s_bbox[0] + 60, s_bbox[3] - s_bbox[1] + 30
    draw.rectangle([(w-bw)//2, y, (w+bw)//2, y + bh], outline=(180, 180, 180), width=2)
    draw.text(((w-bw)//2 + 30, y + 12), stats, font=fonts['stats'], fill=(130, 130, 130))

    y += bh + 60
    sub_text = data.get('subtitle') or ''
    for line in visual_wrap(sub_text, fonts['sub'], w - 200):
        bbox = draw.textbbox((0, 0), line, font=fonts['sub'])
        draw.text(((w - (bbox[2] - bbox[0])) // 2, y), line, font=fonts['sub'], fill=(80, 80, 80))
        y += (bbox[3] - bbox[1]) + 20

    # 🌟 核心修复：在这里强制增加 80 像素的绝对留白空间
    y += 80

    try:
        photo = Image.open(photo_path).convert('RGB')
        new_height = int(photo.height * (w / photo.width))
        photo = photo.resize((w, new_height), Image.Resampling.LANCZOS)

        # 🌟 核心修复：比较「画布底端往上算的高度」和「文字加上留白后的高度」
        # 强迫图片 Y 轴绝对不可以小于文字区域，避免盖字
        paste_y = max(y, h - new_height)

        canvas.paste(photo, (0, paste_y))
    except Exception: pass
    return canvas

def create_inner_canvas(fonts, page_num, total_pages):
    w, h = 1080, 1440
    canvas = Image.new('RGB', (w, h), color=(250, 250, 250))
    draw = ImageDraw.Draw(canvas)
    margin = 80
    draw.text((margin, 60), "LUCAS 拆书实验室 | 核心观点", font=fonts['footer'], fill=(150, 150, 150))
    draw.rectangle([margin, 130, margin + 60, 138], fill=(230, 50, 50))
    p_str = f"PAGE {page_num:02d} / {total_pages:02d}"
    p_bbox = draw.textbbox((0, 0), p_str, font=fonts['footer'])
    draw.text((w - margin - (p_bbox[2]-p_bbox[0]), 60), p_str, font=fonts['footer'], fill=(180, 180, 180))
    return canvas, draw

def generate_slides_from_data(text_data, photo_path, font_dict_ignored):
    """兼容新版中控台的主工厂函数 (🌟 终极清洗 + 分类加强版)"""
    setup_environment()

    fonts = {
        'title': ImageFont.truetype(BOLD_PATH, 115, index=2),
        'title_s': ImageFont.truetype(BOLD_PATH, 48, index=2),
        'sub': ImageFont.truetype(BOLD_PATH, 42, index=2),
        'stats': ImageFont.truetype(FONT_PATH, 40, index=2),
        'body': ImageFont.truetype(FONT_PATH, 45, index=2),
        'footer': ImageFont.truetype(FONT_PATH, 32, index=2)
    }

    raw_paras = text_data.get('content', '').replace('\r\n', '\n').replace('**', '').strip().split('\n')
    processed = []

    for p in raw_paras:
        p = p.strip()
        if not p: continue

        m_mag = re.match(r'^(\d{1,2})\s*[\/|\-]\s*(.*)', p)
        m_num = re.match(r'^(\d+)[\.、]\s*(.*)', p)
        m_cn = re.match(r'^([一二三四五六七八九十]+)[\.、]\s*(.*)', p)
        m_bracket = re.match(r'^([【\[])(.*?)([】\]])(.*)', p)

        is_short_no_period = len(p) < 40 and not p.endswith('。') and not p.endswith('！')

        if m_mag:
            processed.append({'type': 'sub_heading', 'text': f"{m_mag.group(1).zfill(2)} / {m_mag.group(2).strip()}"})
        elif m_num and is_short_no_period:
            clean_title = m_num.group(2).replace('**', '').replace('*', '').strip()
            processed.append({'type': 'sub_heading', 'text': f"{m_num.group(1).zfill(2)} / {clean_title}"})
        elif m_num:
            clean_text = m_num.group(2).replace('**', '').replace('*', '').replace('#', '').strip()
            processed.append({'type': 'num_list', 'num': m_num.group(1), 'text': clean_text})
        elif p.startswith('-') or p.startswith('✦') or p.startswith('■') or p.startswith('•'):
            processed.append({'type': 'bullet_list', 'text': p[1:].strip()})
        elif m_bracket:
            clean_bracket_text = m_bracket.group(2).replace('**', '').replace('*', '').replace('#', '').strip()
            processed.append({'type': 'sub_heading', 'text': f"▶ / {clean_bracket_text}"})
        else:
            clean_para_text = p.replace('**', '').replace('*', '').replace('#', '').strip()
            processed.append({'type': 'para', 'text': clean_para_text})

    paginated = paginate_content(processed, fonts)
    total_imgs = len(paginated) + 1
    gen_files = []

    c_img = draw_cover(text_data.get('cover', {}), photo_path, fonts)
    c_name = f"Slide_01.jpg"
    c_img.save(c_name, quality=100)
    gen_files.append(c_name)

    for i, p_lines in enumerate(paginated):
        canvas, draw = create_inner_canvas(fonts, i + 2, total_imgs)
        margin = 80
        for lo in p_lines:
            if lo['type'] == 'sub_heading':
                draw.text((margin, lo['y']), lo['text'], font=fonts['title_s'], fill=(200, 40, 40))
            elif lo['type'] == 'num_list':
                if lo['is_first_line']: draw.text((margin, lo['y']), lo['prefix'], font=fonts['title_s'], fill=(230, 50, 50))
                draw.text((margin + lo['indent'], lo['y']), lo['text'], font=fonts['body'], fill=(50, 50, 50))
            elif lo['type'] == 'bullet_list':
                if lo['is_first_line']: draw.rectangle([margin, lo['y']+22, margin+14, lo['y']+36], fill=(230, 50, 50))
                draw.text((margin + lo['indent'], lo['y']), lo['text'], font=fonts['body'], fill=(50, 50, 50))
            else:
                draw.text((margin, lo['y']), lo['text'], font=fonts['body'], fill=(50, 50, 50))
        name = f"Slide_{i+2:02d}.jpg"
        canvas.save(name, quality=100)
        gen_files.append(name)

    return gen_files

## 文字引擎

In [ ]:
!pip install duckduckgo-search

In [ ]:
# ==========================================
# 🧠 模块五：核心 AI 引擎 (RSS选題 + DuckDuckGo肉搜 + 长文生成)
# ==========================================
import urllib.parse
import xml.etree.ElementTree as ET
import requests
import json
import re
from duckduckgo_search import DDGS

# 2026 官方免费版模型矩阵
FALLBACK_MODELS = [
    "gemini-3.1-flash-lite",
    "gemini-3-flash-preview",
    "gemini-2.5-flash",
    "gemini-2.5-flash-lite",
    "gemini-2.5-pro"
]

# ---------------------------------------------------------
# 【外挂搜索引擎】
# ---------------------------------------------------------
def custom_web_search(keyword):
    print(f"🌍 启动搜索：{keyword}")
    try:
        results = DDGS().text(keyword, max_results=3)
        context = ""
        for idx, r in enumerate(results):
            context += f"【来源 {idx+1}】: {r['title']}\n【摘要】: {r['body']}\n\n"
        return context if context else "未搜寻到相关数据。"
    except Exception as e:
        print(f"搜索异常: {e}")
        return "搜索功能暂时离线。"

# ---------------------------------------------------------
# 【STEP 0：雷达选题生成器 (基于 RSS)】
# ---------------------------------------------------------
def fetch_viral_topics(api_key, radar_mode):
    mode_config = {
        "🔥 搞钱与职场生存": {"kw": "搞钱 OR 职场 OR 裁员 OR 就业 OR 经济", "focus": "搞钱搞事业、职场生存法则"},
        "🧠 认知觉醒与学习": {"kw": "教育 OR 考研 OR 留学 OR 青年 OR 心理", "focus": "高效学习法、认知升级"},
        "📸 生活美学与爱好": {"kw": "健康 OR 运动 OR 摄影 OR 数码", "focus": "硬核爱好、极简生活"},
        "❤️ 情感洞察与关系": {"kw": "恋爱 OR 婚姻 OR 社交 OR 情绪", "focus": "高价值社交、人性博弈"}
    }

    current_config = mode_config.get(radar_mode, mode_config["🔥 搞钱与职场生存"])
    encoded_query = urllib.parse.quote(current_config["kw"])

    rss_url = f"https://news.google.com/rss/search?q={encoded_query}+when:7d&hl=zh-CN&gl=CN&ceid=CN:zh-Hans"
    real_hot_events = ""
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(rss_url, headers=headers, timeout=8)
        response.raise_for_status()
        root = ET.fromstring(response.content)
        items = root.findall('.//item')
        if not items: raise ValueError("未能获取新闻")
        top_15_titles = [item.find('title').text for item in items[:15]]
        real_hot_events = "、".join(top_15_titles)
        print(f"📡 拦截新闻成功，共 {len(items)} 条。")
    except Exception as e:
        raise Exception(f"热点爬取失败 ({e})")

    prompt = f"""
    【真实新闻】：{real_hot_events}
    请从中挑选 5 个事件，转化为小红书爆款选题，紧扣【{current_config["focus"]}】方向。
    必须借势真实新闻！严格输出 JSON 数组格式（仅包含5个字符串），禁止 Markdown。
    """

    payload = {
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"responseMimeType": "application/json"},
        "safetySettings": [
            {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"}
        ]
    }

    for model_name in FALLBACK_MODELS:
        url = f"https://generativelanguage.googleapis.com/v1beta/models/{model_name}:generateContent?key={api_key}"
        try:
            res = requests.post(url, headers={"Content-Type": "application/json"}, json=payload, timeout=30).json()
            if 'error' in res or 'candidates' not in res: continue

            raw_text = res['candidates'][0]['content']['parts'][0]['text']
            match = re.search(r'\[\s*".*?"\s*\]', raw_text, re.DOTALL)
            clean_text = match.group(0) if match else raw_text.replace("```json", "").replace("```", "").strip()

            topics = json.loads(clean_text)
            if topics: return topics, real_hot_events
        except Exception: continue

    raise Exception("所有模型均已失效，请稍后再试。")

# ---------------------------------------------------------
# 【STEP 1：深度长文生成器 (基于 DuckDuckGo 肉搜)】
# ---------------------------------------------------------
def fetch_text_options(api_key, title, scraped_data=""):
    # 根据用户选定的题目，执行精准搜索
    search_keyword = f"{title} 真实事件细节 OR 最新现状 OR 核心争议"
    real_world_data = custom_web_search(search_keyword)

    prompt = f"""
    【背景设定】：你是一个极度犀利的人性观察家、实战派操盘手与顶级知识博主。你精通职场、学习、生活与情感领域的深度分析，极其讨厌“假大空”的爹味说教。

    【🔥 真实情报来源】：请**绝对依赖**以下真实数据与网页摘要！
    👇👇👇
    {real_world_data}
    👆👆👆

    【输入标题】：{title}

    【写作任务】：基于搜索到的真实情报，输出 3 个版本的小红书深度长文（1000字左右），并为每个版本构思适合极简冷峻风格的英文画面提示词（供 Stable Diffusion 渲染使用）。

    【排版生死线】：
    1. 彻底封杀 AI 味词汇：绝对禁止出现“不可否认、瞬息万变、底层逻辑、赋能、总而言之”。
    2. 小标题只能用：「01 / 标题名称」
    3. 列表重点只能用：「- 内容」
    4. 绝对禁止使用 `**`、`#` 等 Markdown 符号！
    5. 标题必须在12字以内，禁带#号和括号！废话全塞进 subtitle 里。
    6. 极致呼吸感：使用 \\n\\n 频繁换行，每段不超过 80 字。

    请输出 JSON 数组格式（严格遵守）：
    [
        {{
            "version_name": "硬核客观拆解版",
            "cover": {{"title": "（12字内极短句）", "subtitle": "（直击痛点的副标题）"}},
            "content": "（🔥写作指令【路线B】：拒绝捏造个人故事！直接从【情报数据】中提取真实的机构、社会现象、研究报告或客观统计数据作为开局。像外科医生一样拆解这个现象/事件背后的真相。用具体的客观数据来算一笔极度真实的“账”（根据话题决定是算经济账、时间账、还是情绪损耗账）。确保使用 \\n\\n 换行，小标题严格使用 01 / 格式）",
            "tags": "（根据当前话题生成10个垂直领域的精准标签）",
            "image_prompts": [
                "（🔥生图指令：根据本篇文章的核心意象，生成纯英文的画面主体描述，突出冷酷客观的氛围，如 a close up of a cracked smartphone screen showing alarming data）",
                "（纯英文画面主体描述2，聚焦物品或局部细节）",
                "（纯英文画面主体描述3）"
            ]
        }},
        {{
            "version_name": "沉浸式人间故事版",
            "cover": {{"title": "（12字内扎心短句）", "subtitle": "（情绪铺垫的副标题）"}},
            "content": "（🔥写作指令【路线A】：基于情报中的宏观现状，构建一个『最容易被该现象波及的典型人物画像』。给他/她设定具体的身份特征（明确写出“化名老李，32岁，XX身份...”以示客观代表性）。用这个人物的第一人称视角，描绘被现实毒打的困境。开篇必须是一个具体的动作或对话，描写具体的物品（如冷掉的外卖、揉皱的考研资料、没回的微信）。弱化上帝视角，极具活人感！确保使用 \\n\\n 换行，小标题严格使用 01 / 格式）",
            "tags": "（根据当前话题生成10个情感共鸣或生活记录类标签）",
            "image_prompts": [
                "（🔥生图指令：根据本篇的私密日记感，生成纯英文画面描述，如 a half empty iced americano on a messy desk with scattered notes）",
                "（纯英文画面主体描述2，如深夜发光的屏幕局部）",
                "（纯英文画面主体描述3）"
            ]
        }},
        {{
            "version_name": "硬核逻辑推演版",
            "cover": {{"title": "（12字内行动短语）", "subtitle": "（痛点点出与解法预告）"}},
            "content": "（🔥写作指令：拒绝裸奔的道理！必须抛出情报中的真实事实作为论据，实行【观点+真实微操举例】的绝对公式。每次抛出概念，必须结合情报给出接地气的实操例子。必须有理论支撑，且落地到普通人的日常选择上。确保使用 \\n\\n 换行，小标题严格使用 01 / 格式）",
            "tags": "（根据当前话题生成10个干货或方法论类标签）",
            "image_prompts": [
                "（🔥生图指令：根据本篇的硬核逻辑感，生成纯英文画面描述，如 minimalist geometric shadows on a concrete wall）",
                "（纯英文画面主体描述2，冷酷理性的意象）",
                "（纯英文画面主体描述3）"
            ]
        }}
    ]
    """

    payload = {
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"responseMimeType": "application/json"},
        "safetySettings": [
            {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"}
        ]
    }

    for model_name in FALLBACK_MODELS:
        print(f"🧠 调用模型【{model_name}】生成长文...")
        url = f"https://generativelanguage.googleapis.com/v1beta/models/{model_name}:generateContent?key={api_key}"
        try:
            res = requests.post(url, headers={"Content-Type": "application/json"}, json=payload, timeout=90).json()
            if 'error' in res or 'candidates' not in res: continue

            raw_text = res['candidates'][0]['content']['parts'][0]['text']
            match = re.search(r'\[\s*\{.*?\}\s*\]', raw_text, re.DOTALL)
            if not match: continue

            data = json.loads(match.group(0))
            if isinstance(data, list):
                for item in data:
                    cover = item.get('cover', {})
                    raw_title = cover.get('title', '').replace('#', '').replace('【', '').replace('】', '')
                    cover['title'] = raw_title[:14].strip()

                    raw_content = item.get('content', '').replace('**', '').replace('*', '')
                    item['content'] = re.sub(r'(?m)^#+\s*', '', raw_content).strip()

                    content_length = len(item['content'])
                    cover['words'] = str(content_length)
                    cover['time'] = f"{max(1, content_length // 250)} mins"
                    item['cover'] = cover
                return data
        except Exception: continue

    return []

## 圖片引擎

In [ ]:
!pip install diffusers transformers accelerate

In [ ]:
# ==========================================
# 🖼️ 图片模块：本地 GPU 算力生图引擎 (🌟 终极去色版)
# ==========================================
import torch
from diffusers import AutoPipelineForText2Image, DPMSolverMultistepScheduler, AutoencoderKL
from PIL import Image

# 全局变量，用于保持模型常驻显存
_local_sd_pipe = None

def load_sd_engine():
    """首次运行加载模型引擎"""
    global _local_sd_pipe
    device = "cuda" if torch.cuda.is_available() else "cpu"

    if _local_sd_pipe is None:
        print(f"\n🖼️ 正在初始化本地 SD 生图引擎 (显卡: {device.upper()})...")
        print("   📥 首次运行：正在拉取色彩解码器与超写实专业模型，请稍候...")

        try:
            # 1. 加载 VAE 色彩解码器
            vae = AutoencoderKL.from_pretrained(
                "stabilityai/sd-vae-ft-mse",
                torch_dtype=torch.float16 if device == "cuda" else torch.float32
            )

            # 2. 加载写实风主力模型 (SG161222/Realistic_Vision_V5.1_noVAE)
            _local_sd_pipe = AutoPipelineForText2Image.from_pretrained(
                "SG161222/Realistic_Vision_V5.1_noVAE",
                vae=vae,
                torch_dtype=torch.float16 if device == "cuda" else torch.float32
            )
            _local_sd_pipe = _local_sd_pipe.to(device)

            # 3. 性能与算法优化
            _local_sd_pipe.safety_checker = None # 物理关闭 NSFW 检查，节省算力
            _local_sd_pipe.scheduler = DPMSolverMultistepScheduler.from_config(
                _local_sd_pipe.scheduler.config,
                use_karras_sigmas=True, # 启用 Karras 算法，大幅减少步数
                algorithm_type="dpmsolver++"
            )

            # 4. (可选) xformers 优化，如果未安装会导致启动警告，不影响使用
            try:
                _local_sd_pipe.enable_xformers_memory_efficient_attention()
                print("   🚀 已启用 xformers 内存优化")
            except Exception:
                pass

            print("   ✅ SD 引擎加载完毕！")

        except Exception as e:
            print(f"   ❌ 模型加载失败 (检查算力或网络): {e}")
            return False

    return True

def generate_grayscale_images(prompts_list):
    """
    输入：从文字引擎获取的 3 句纯英文生图提示词列表
    过程：SD 渲染 -> PIL 物理去色 -> 保存
    回传：保存后的 3 张图片文件名列表
    """
    global _local_sd_pipe

    if not load_sd_engine():
        return []

    if not prompts_list or len(prompts_list) < 3:
        print("   ❌ 未收到有效的 3 句 AI 提示词。")
        return []

    # 严格切片，只处理前 3 句
    final_prompts = prompts_list[:3]

    # === 挂载黑白真实感滤镜 (🌟 终极去色魔法的一部分) ===
    style_prefix = "RAW photo, "
    style_suffix = ", amateur photography, Fujifilm XT4, unedited, candid, highly realistic, minimalist, harsh directional sunlight, heavy dramatic shadows, film grain, 8k uhd, highly detailed, moody atmospheric"

    # 增强型黑白反向提示词，物理层面封杀色彩词汇
    neg_prompt = "(deformed iris, deformed pupils, semi-realistic, cgi, 3d, render, sketch, cartoon, drawing, anime:1.4), artificial light, oversharpened, plastic, color, vibrant, colorful, sepia, rgb, tint, hue, text, close up, cropped, worst quality, low quality, jpeg artifacts, ugly, duplicate, morbid"

    print(f"\n🖼️ 启动 GPU 算力生图 (物理生图 steps: 40, CFG: 5.5)...")
    generated_images_files = []

    for i, p in enumerate(final_prompts):
        # 1. 清洗并组合最终生图咒语
        clean_p = p.strip()
        # 移除 AI 可能误读加上的 raw photo 词汇，避免前缀重复
        clean_p = clean_p.replace("RAW photo,", "").strip()
        final_prompt = f"{style_prefix}{clean_p}{style_suffix}"

        print(f"   ▶️ 正在渲染单色图 {i+1} / 3：({clean_p[:40]}...)")

        try:
            # 2. 调用 SD 引擎生成
            image = _local_sd_pipe(
                prompt=final_prompt,
                negative_prompt=neg_prompt,
                num_inference_steps=40, # 使用 DPMSolver+Karras 40 步即可获得神级质感
                width=768,
                height=512, # 小红书常用宽图比例
                guidance_scale=5.5 # CFG 保持在 5.5，避免过度写实导致的虚假感
            ).images[0]

            # 3. 🌟 終極物理去色：用 Python 强行剥夺所有颜色，转换为纯灰阶，再转回 RGB 排版
            image = image.convert("L").convert("RGB")

            # 4. 保存图片
            img_name = f"photo_v{i+1}.jpg"
            image.save(img_name, quality=100) # 满质量保存
            generated_images_files.append(img_name)
            print(f"   ✅ 图片 {i+1} 渲染并物理去色完成！")

        except Exception as e:
            print(f"   ❌ 图片 {i+1} 渲染出错: {e}")

    return generated_images_files

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


## 可視化控制台

In [ ]:
# ==========================================
# 🎛️ 模块六：决策者中控台 (🌟 一键打包 ZIP 下载版)
# ==========================================
import ipywidgets as widgets
from IPython.display import display, clear_output
from IPython.display import Image as IPImage
import zipfile
import os

def main_dashboard():
    clear_output()
    print("="*60)
    print("      LUCAS 拆书实验室 | 终极灵魂图文一体中控台")
    print("="*60)

    title_input = widgets.Text(
        value='',
        description='📝 标题：', layout=widgets.Layout(width='90%')
    )
    btn_step1 = widgets.Button(
        description="【第 1 步】一键生成图文联动素材", button_style='primary',
        layout=widgets.Layout(width='300px', height='45px', margin='10px 0 20px 0')
    )

    out_text_preview = widgets.Output()
    text_selector = widgets.Dropdown(options=['(请先生成素材)'], description='✍️ 选定文案：', disabled=True, layout=widgets.Layout(width='80%'))
    btn_regen_text = widgets.Button(description="🔄 独立重写文案", button_style='warning', disabled=True, layout=widgets.Layout(width='150px'))

    out_photo_preview = widgets.Output()
    photo_selector = widgets.Dropdown(options=['(请先生成素材)'], description='🖼️ 选定图片：', disabled=True, layout=widgets.Layout(width='80%'))
    btn_regen_photo = widgets.Button(description="🔄 独立重画图片", button_style='info', disabled=True, layout=widgets.Layout(width='150px'))

    btn_step2 = widgets.Button(
        description="【第 2 步】确认合并 & 渲染排版", button_style='danger', icon='magic', disabled=True,
        layout=widgets.Layout(width='350px', height='55px', margin='20px 0 10px 0')
    )
    out_render = widgets.Output()

    # 🌟 更新为打包下载按钮
    btn_step3 = widgets.Button(
        description="【第 3 步】一键打包下载最终图文 (ZIP)", button_style='success', icon='download', disabled=True,
        layout=widgets.Layout(width='300px', height='50px', margin='10px 0 0 0')
    )

    store = {'texts': [], 'photos': [], 'files': []}

    def render_texts():
        with out_text_preview:
            clear_output()
            if not store['texts']: return print("⚠️ 文案获取失败。")
            print("="*20 + " 文案预览区 " + "="*20)
            text_opts = []
            for i, t in enumerate(store['texts']):
                v_name = t.get('version_name', f'版本 {i+1}')
                print(f"【选项 {i+1}：{v_name}】")
                print(f"🏷️ 专属标签：{t.get('tags', '无标签')}")
                print(f"🎨 专属图意：{t.get('image_prompt', '未生成')[:50]}...")
                print(f"封面标题：{t.get('cover', {}).get('title', '无标题')}")
                print(f"正文预览：{t.get('content', '')[:80]}...\n")
                text_opts.append((f"选项 {i+1}：{v_name}", i))
            text_selector.options = text_opts
            text_selector.disabled = False; btn_regen_text.disabled = False

    def render_photos():
        with out_photo_preview:
            clear_output()
            if not store['photos']: return print("⚠️ 图片获取失败。")
            print("="*20 + " 文章专属配图预览区 " + "="*20)
            img_widgets = []
            photo_opts = []
            for i, p_path in enumerate(store['photos']):
                with open(p_path, "rb") as f:
                    img_widgets.append(widgets.Image(value=f.read(), format='jpg', width=220))
                photo_opts.append((f"使用 图片 {i+1}", i))
            display(widgets.HBox(img_widgets))
            photo_selector.options = photo_opts
            photo_selector.disabled = False; btn_regen_photo.disabled = False

    def get_prompts_from_texts():
        if not store['texts']: return None
        return [t.get('image_prompt', "RAW photo, a highly realistic modern minimalist desk, cinematic black and white, 8k") for t in store['texts']]

    def on_step1_clicked(b):
        btn_step1.description = "⏳ 正在拼命请求中..."
        btn_step1.disabled = True
        with out_text_preview: clear_output(); print("🚀 正在请求大腦思考文章与画面...")
        # 确保之前填过 GOOGLE_API_KEY
        store['texts'] = fetch_text_options(GOOGLE_API_KEY, title_input.value)
        render_texts()

        with out_photo_preview: clear_output(); print("🚀 显卡正在根据文章内容作画...")
        prompts = get_prompts_from_texts()
        store['photos'] = fetch_photo_options(prompts)
        render_photos()

        btn_step1.description = "【第 1 步】全部重新生成"
        btn_step1.disabled = False; btn_step2.disabled = False

    def on_regen_text_clicked(b):
        btn_regen_text.description = "⏳ 正在重写..."
        btn_regen_text.disabled = True
        with out_text_preview: clear_output(); print("🚀 正在重新构思文案与画面指令...")
        store['texts'] = fetch_text_options(GOOGLE_API_KEY, title_input.value)
        render_texts()
        btn_regen_text.description = "🔄 独立重写文案"

    def on_regen_photo_clicked(b):
        btn_regen_photo.description = "⏳ 正在重画..."
        btn_regen_photo.disabled = True
        with out_photo_preview: clear_output(); print("🚀 正在依据当前文章指令重新渲染图片...")
        prompts = get_prompts_from_texts()
        store['photos'] = fetch_photo_options(prompts)
        render_photos()
        btn_regen_photo.description = "🔄 独立重画图片"

    def on_step2_clicked(b):
        with out_render:
            clear_output(); print("⚙️ 正在执行像素级排版引擎，请稍候...")
            try:
                selected_text_raw = store['texts'][text_selector.value]
                text_for_image = selected_text_raw.copy()
                tags_str = text_for_image.pop('tags', '')
                text_for_image.pop('image_prompt', '')

                selected_photo = store['photos'][photo_selector.value]
                store['files'] = generate_slides_from_data(text_for_image, selected_photo, None)

                print("✅ 排版完成！最终效果如下：")
                for f in store['files']: display(IPImage(f, width=300))

                print("\n" + "="*50)
                display(widgets.HTML("<h3>📋 小红书发布面板 (点击框内全选复制)</h3>"))
                display(widgets.VBox([
                    widgets.Text(value=selected_text_raw.get('cover', {}).get('title', '无标题'), description="📌 标题:", layout=widgets.Layout(width='90%')),
                    widgets.Textarea(value=selected_text_raw.get('content', '无正文内容'), description="📝 正文:", layout=widgets.Layout(width='90%', height='250px')),
                    widgets.Textarea(value=tags_str, description="🏷️ 标签:", layout=widgets.Layout(width='90%', height='60px'))
                ]))
                print("="*50)
                btn_step3.disabled = False
            except Exception as e: print(f"❌ 排版失败：{e}")

    # 🌟 突破 10 张图下载限制的核心逻辑
    def on_step3_clicked(b):
        print("🚀 正在生成无视浏览器拦截的「直连下载通道」...")
        try:
            import base64
            import os
            from IPython.display import display, HTML

            html_str = "<div style='margin-top: 20px; padding: 20px; border: 2px dashed #ccc; border-radius: 10px;'>"
            html_str += "<h3>👇 高清原图已备妥，请点击下方按钮分别下载（单点单下，绝对不挡）：</h3>"
            html_str += "<div style='display: flex; flex-wrap: wrap; gap: 15px; margin-top: 15px;'>"

            for idx, f in enumerate(store['files']):
                if os.path.exists(f):
                    with open(f, "rb") as img:
                        b64 = base64.b64encode(img.read()).decode()

                    # 生成精美的原生 HTML 下载按钮
                    btn_html = f"""
                    <a href="data:image/jpeg;base64,{b64}" download="{f}"
                       style="padding: 12px 24px; background-color: #28a745; color: white; text-decoration: none;
                              border-radius: 6px; font-weight: bold; text-align: center; box-shadow: 0 4px 6px rgba(0,0,0,0.1); transition: 0.2s;">
                       📥 下载 {f}
                    </a>
                    """
                    html_str += btn_html

            html_str += "</div></div>"

            # 直接渲染出下载控制台
            display(HTML(html_str))

            btn_step3.description = "✅ 下载通道已生成在下方！"
            btn_step3.button_style = 'success'
        except Exception as e:
            print(f"❌ 通道生成失败：{e}")
    btn_step1.on_click(on_step1_clicked)
    btn_regen_text.on_click(on_regen_text_clicked)
    btn_regen_photo.on_click(on_regen_photo_clicked)
    btn_step2.on_click(on_step2_clicked)
    btn_step3.on_click(on_step3_clicked)

    text_box = widgets.VBox([out_text_preview, widgets.HBox([text_selector, btn_regen_text])])
    photo_box = widgets.VBox([out_photo_preview, widgets.HBox([photo_selector, btn_regen_photo])])

    # 🌟 就是这一行 display 把介面印出来的，刚刚一定是不小心删到它了
    display(widgets.VBox([
        title_input, btn_step1, text_box, widgets.HTML("<hr>"), photo_box, widgets.HTML("<hr>"), btn_step2, out_render, btn_step3
    ]))

if __name__ == "__main__":
    main_dashboard()

      LUCAS 拆书实验室 | 终极灵魂图文一体中控台


### Gradio version


In [ ]:
# ==========================================
# 📱 模块七：Gradio 旗舰手机端 App (V9.6 全方位波段雷达版 + 数据无缝传导)
# ==========================================
import gradio as gr
import urllib.parse
import os
import shutil

# 🌟 旗舰级 CSS 美化
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Noto+Sans+SC:wght@400;700&display=swap');
body, * { font-family: 'Noto Sans SC', sans-serif !important; }
.main-container { max-width: 950px !important; margin: 0 auto; padding: 20px; }
.header-box { text-align: center; padding: 35px 20px; background: linear-gradient(135deg, #1f1f1f 0%, #3d3d3d 100%); color: white; border-radius: 16px; margin-bottom: 30px; box-shadow: 0 10px 25px rgba(0,0,0,0.2); }
.header-box h1 { margin: 0; font-weight: 700; font-size: 2.2em; color: #fff; letter-spacing: 2px; }
.header-box p { margin-top: 10px; color: #ccc; font-size: 1.1em; }
.step-title { font-size: 1.4em; font-weight: bold; margin: 25px 0 15px 0; color: #222; border-left: 5px solid #E63232; padding-left: 12px; }
.card { background: white; border-radius: 14px; padding: 20px; box-shadow: 0 4px 15px rgba(0,0,0,0.05); border: 1px solid #eaeaea; transition: all 0.3s ease; height: 100%; display: flex; flex-direction: column;}
.card:hover { transform: translateY(-4px); box-shadow: 0 12px 25px rgba(0,0,0,0.1); border-color: #E63232; }
.btn-primary { background: #E63232 !important; color: white !important; font-weight: bold !important; border: none !important; box-shadow: 0 4px 10px rgba(230, 50, 50, 0.3) !important; transition: all 0.2s !important; }
.btn-primary:hover { background: #cc2a2a !important; transform: scale(1.02); }
.btn-secondary { border: 2px solid #E63232 !important; color: #E63232 !important; background: transparent !important; font-weight: bold !important; margin-top: auto !important; transition: all 0.2s !important;}
.btn-secondary:hover { background: #E63232 !important; color: white !important; }
.btn-refresh { border: 1px dashed #bbb !important; color: #666 !important; background: transparent !important; font-weight: bold !important; transition: all 0.2s !important; }
.btn-refresh:hover { border-color: #E63232 !important; color: #E63232 !important; background: #fff5f5 !important; }
hr { border-color: #f0f0f0; margin: 30px 0; }
"""

# ==========================================
# 交互包装函数
# ==========================================
def process_step0_topics(radar_mode):
    if 'GOOGLE_API_KEY' not in globals() or not GOOGLE_API_KEY:
        raise gr.Error("⚠️ 请先在前面的储存格填写 GOOGLE_API_KEY！")

    gr.Info(f"📡 正在全网扫描【{radar_mode}】频道热点...")
    # 💡 接收两个返回值
    topics, raw_scraped_data = fetch_viral_topics(GOOGLE_API_KEY, radar_mode)
    gr.Info("✅ 找到 5 个高潜力痛点，请在下方点击选择！")
    # 把 raw_scraped_data 一并传递出去给状态池
    return gr.update(choices=topics, value=topics[0], visible=True), topics[0], raw_scraped_data

def update_input_title(selected_topic):
    return selected_topic

# 💡 核心修复：新增 scraped_data 变量接收来自雷达的数据
def process_step1_text(title, scraped_data):
    if 'GOOGLE_API_KEY' not in globals() or not GOOGLE_API_KEY:
        raise gr.Error("⚠️ 请先填写 GOOGLE_API_KEY！")

    gr.Info("🧠 正在构思三款爆款文案，请稍候...")
    data = fetch_text_options(GOOGLE_API_KEY, title, scraped_data)
    if not data or len(data) < 3:
        raise gr.Error("⚠️ 文案生成失败，请重试")

    # 🔪 终极断头台：在展示预览前，直接修改核心数据，把超长标题强行切断！
    for item in data:
        raw_t = item.get('cover', {}).get('title', '')
        # 1. 如果 AI 偷塞了 #hashtag，直接切掉 # 之后的所有东西
        if '#' in raw_t:
            raw_t = raw_t.split('#')[0].strip()

        # 2. 强制 18 字物理截断 (预留一点标点符号空间，保证绝对不超过20)
        if len(raw_t) > 18:
            raw_t = raw_t[:18]

        # 把清洗干净的标题存回资料库
        item['cover']['title'] = raw_t

    # 渲染预览画面
    previews = []
    for i in range(3):
        v_name = data[i].get('version_name', f'版本 {i+1}')
        title_text = data[i].get('cover', {}).get('title', '无标题')
        content_preview = data[i].get('content', '')[:100]
        previews.append(f"### {v_name}\n**{title_text}**\n\n<span style='color:#666;'>{content_preview}...</span>")

    return previews[0], previews[1], previews[2], data
def process_step2_images(idx, all_data):
    selected = all_data[int(idx)]
    prompts = selected.get('image_prompts', [])
    if not isinstance(prompts, list) or len(prompts) < 3:
        base_prompt = selected.get('image_prompt', 'a broken clock')
        prompts = [
            f"extreme close up shot of {base_prompt}, macro photography",
            f"wide angle shot of {base_prompt}, cinematic wide view",
            f"top down view of {base_prompt}, flat lay"
        ]

    gr.Info("🎨 正在全力渲染 3 款不同构图的 8K 大图...")
    photo_paths = fetch_photo_options(prompts[:3])

    if not photo_paths:
        raise gr.Error("图片渲染失败！")

    p1 = photo_paths[0] if len(photo_paths) > 0 else None
    p2 = photo_paths[1] if len(photo_paths) > 1 else p1
    p3 = photo_paths[2] if len(photo_paths) > 2 else p1

    gr.Info("✅ 图片生成完毕！请在下方挑选一张你最喜欢的。")
    return p1, p2, p3, selected

def process_regenerate_images(selected_data):
    if not selected_data:
        raise gr.Error("⚠️ 请先在第一步选定一款文案！")
    prompts = selected_data.get('image_prompts', [])
    if not isinstance(prompts, list) or len(prompts) < 3:
        base_prompt = selected_data.get('image_prompt', 'a broken clock')
        prompts = [base_prompt] * 3

    gr.Info("🔄 正在重新渲染 3 款全新视角的底图...")
    regen_prompts = [f"completely different variation, {p}" for p in prompts[:3]]
    photo_paths = fetch_photo_options(regen_prompts)

    p1 = photo_paths[0] if len(photo_paths) > 0 else None
    p2 = photo_paths[1] if len(photo_paths) > 1 else p1
    p3 = photo_paths[2] if len(photo_paths) > 2 else p1
    gr.Info("✅ 新图片生成完毕！")
    return p1, p2, p3

def process_step3_render(img_path, selected_data):
    if not img_path:
        raise gr.Error("⚠️ 请先选择一张图片！")
    gr.Info("📐 正在启动像素级排版引擎...")
    slides = generate_slides_from_data(selected_data, img_path, None)

    saved_files = []
    for i, slide in enumerate(slides):
        file_name = f"LUCAS_Slide_Page_{i+1}.jpg"
        if isinstance(slide, str) and os.path.exists(slide):
            import shutil
            shutil.copyfile(slide, file_name)
            saved_files.append(file_name)
        else:
            slide.save(file_name, format="JPEG", quality=95)
            saved_files.append(file_name)

    # 🛡️ 强制 20 字截断防呆机制
    raw_title = selected_data.get('cover', {}).get('title', '')
    if len(raw_title) > 20:
        final_title_text = raw_title[:20]
        print(f"⚠️ 警告：AI 标题超字，已自动截断为 20 字 -> {final_title_text}")
    else:
        final_title_text = raw_title

    content_text = selected_data.get('content', '')
    tags_text = selected_data.get('tags', '')

    gr.Info("🎉 恭喜！高清海报已完成。可以在下方列表挑选下载。")
    return slides, saved_files, final_title_text, content_text, tags_text

def hide_warning():
    return None

# ==========================================
# 搭建旗舰 UI 界面
# ==========================================
with gr.Blocks(css=custom_css, theme=gr.themes.Default()) as demo:
    all_data_store = gr.State([])
    selected_text_store = gr.State({})
    # 💡 核心修复：建立爬虫数据专用金库，随时等候被调用
    raw_scraped_data_store = gr.State("")

    with gr.Column(elem_classes="main-container"):
        with gr.Column(elem_classes="header-box"):
            gr.Markdown("<h1>LUCAS 拆书实验室</h1><p>全自动小红书爆款图文生成舱 V9.6 (多频道雷达版)</p>")

        # 🌟 全新 STEP 0 (多频道波段雷达)
        gr.Markdown("<div class='step-title'>STEP 0. 爆款选题雷达 (多频道抓取)</div>")
        with gr.Row():
            radar_mode_ui = gr.Radio(
                choices=["🔥 搞钱与职场生存", "🧠 认知觉醒与学习", "📸 生活美学与爱好", "❤️ 情感洞察与关系"],
                value="🧠 认知觉醒与学习",
                label="🎯 请选择今天你想聊的频道：",
                interactive=True
            )
        with gr.Row():
            gen_topics_btn = gr.Button("📡 扫描此频道今日热点 ➔ 智能生成 5 个选题", elem_classes="btn-primary")
        with gr.Row():
            topics_radio = gr.Radio(choices=[], label="👇 点击下方任一现象，自动填入下一步：", visible=False, interactive=True)

        gr.HTML("<hr>")

        # 🌟 STEP 1
        gr.Markdown("<div class='step-title'>STEP 1. 构思爆款文案</div>")
        with gr.Row():
            input_title = gr.Textbox(show_label=False, placeholder="可由上方智能填入，也可在此手动输入想法", scale=4, container=False)
            gen_text_btn = gr.Button("🚀 构思 3 款文案", elem_classes="btn-primary", scale=2)
            regen_text_btn = gr.Button("🔄 换一批", elem_classes="btn-refresh", scale=1)

        with gr.Row():
            with gr.Column(elem_classes="card"):
                v1_preview = gr.Markdown("⏳ 等待生成中...")
                pick_v1 = gr.Button("✅ 选定此版，去配图", elem_classes="btn-secondary")
            with gr.Column(elem_classes="card"):
                v2_preview = gr.Markdown("⏳ 等待生成中...")
                pick_v2 = gr.Button("✅ 选定此版，去配图", elem_classes="btn-secondary")
            with gr.Column(elem_classes="card"):
                v3_preview = gr.Markdown("⏳ 等待生成中...")
                pick_v3 = gr.Button("✅ 选定此版，去配图", elem_classes="btn-secondary")

        gr.HTML("<hr>")

        # 🌟 STEP 2
        gr.Markdown("<div class='step-title'>STEP 2. 挑选极写实底图</div>")
        with gr.Row():
            with gr.Column(elem_classes="card"):
                img_choice_1 = gr.Image(label="配图方案 A", type="filepath", interactive=False)
                pick_img_1 = gr.Button("🎨 就用这张！开始排版", elem_classes="btn-primary")
            with gr.Column(elem_classes="card"):
                img_choice_2 = gr.Image(label="配图方案 B", type="filepath", interactive=False)
                pick_img_2 = gr.Button("🎨 就用这张！开始排版", elem_classes="btn-primary")
            with gr.Column(elem_classes="card"):
                img_choice_3 = gr.Image(label="配图方案 C", type="filepath", interactive=False)
                pick_img_3 = gr.Button("🎨 就用这张！开始排版", elem_classes="btn-primary")

        with gr.Row():
            regen_img_btn = gr.Button("🔄 图片都没感觉？重新生成 3 张全新底图", elem_classes="btn-refresh")

        gr.HTML("<hr>")

        # 🌟 STEP 3
        gr.Markdown("<div class='step-title'>STEP 3. 获取最终排版与文案</div>")
        with gr.Row():
            with gr.Column(scale=3, elem_classes="card"):
                final_gallery = gr.Gallery(label="🖼️ 高清海报 (点击图片右上角 ⬇️ 可直接下载原图)", columns=2, height="auto", object_fit="contain")
                download_file = gr.File(label="📥 独立保存区：亦可点击列表右侧 ⬇️ 下载 JPG", file_count="multiple")

            with gr.Column(scale=2, elem_classes="card"):
                final_title = gr.Textbox(label="📌 小红书标题", lines=1, show_copy_button=True)
                final_content = gr.Textbox(label="📝 小红书正文", lines=10, show_copy_button=True)
                final_tags = gr.Textbox(label="🏷️ 推荐标签", lines=2, show_copy_button=True)

    # ==========================================
    # 🔗 绑定事件
    # ==========================================
    # STEP 0
    # 💡 核心修复：按钮点击后，不仅更新界面，还要把爬到的数据存进 raw_scraped_data_store
    gen_topics_btn.click(process_step0_topics, inputs=[radar_mode_ui], outputs=[topics_radio, input_title, raw_scraped_data_store])
    topics_radio.change(update_input_title, inputs=topics_radio, outputs=input_title)

    # STEP 1
    # 💡 核心修复：生成文案时，将 raw_scraped_data_store 作为输入传给模型
    gen_text_btn.click(process_step1_text, inputs=[input_title, raw_scraped_data_store], outputs=[v1_preview, v2_preview, v3_preview, all_data_store])
    regen_text_btn.click(process_step1_text, inputs=[input_title, raw_scraped_data_store], outputs=[v1_preview, v2_preview, v3_preview, all_data_store])

    # STEP 2
    pick_v1.click(hide_warning, outputs=None).then(process_step2_images, inputs=[gr.Number(0, visible=False), all_data_store], outputs=[img_choice_1, img_choice_2, img_choice_3, selected_text_store])
    pick_v2.click(hide_warning, outputs=None).then(process_step2_images, inputs=[gr.Number(1, visible=False), all_data_store], outputs=[img_choice_1, img_choice_2, img_choice_3, selected_text_store])
    pick_v3.click(hide_warning, outputs=None).then(process_step2_images, inputs=[gr.Number(2, visible=False), all_data_store], outputs=[img_choice_1, img_choice_2, img_choice_3, selected_text_store])

    regen_img_btn.click(process_regenerate_images, inputs=selected_text_store, outputs=[img_choice_1, img_choice_2, img_choice_3])

    # STEP 3
    pick_img_1.click(process_step3_render, inputs=[img_choice_1, selected_text_store], outputs=[final_gallery, download_file, final_title, final_content, final_tags])
    pick_img_2.click(process_step3_render, inputs=[img_choice_2, selected_text_store], outputs=[final_gallery, download_file, final_title, final_content, final_tags])
    pick_img_3.click(process_step3_render, inputs=[img_choice_3, selected_text_store], outputs=[final_gallery, download_file, final_title, final_content, final_tags])

demo.launch(share=True, debug=True)

/tmp/ipykernel_26308/3975993988.py:158: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Default()) as demo:
/tmp/ipykernel_26308/3975993988.py:158: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Default()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7e1fe3a85473d391e8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


📡 拦截新闻成功，共 100 条。
🌍 启动搜索：摩根大通大佬谈AI：别怕失业，如何利用技术红利在这个动荡期实现弯道超车？ 真实事件细节 OR 最新现状 OR 核心争议


/tmp/ipykernel_26308/3648908807.py:26: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  results = DDGS().text(keyword, max_results=3)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


🧠 调用模型【gemini-3.1-flash-lite】生成长文...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag